# Datavidia Task-2 v3 — Model / Training Notebook

Notebook ini **menghasilkan model** (dan artefak `.onnx` + `.joblib`) dari data mentah.

Notebook pasangannya: `submission_v3.ipynb` (inference-only, tanpa train ulang).

---

## Pipeline

1. **Patch-Chamfer feature extraction** — DINOv2-S/14 @336px, 24×24 patch, alignment-free chamfer distance pre→post, lalu `pmean_pre`, `pmean_post`, `topchg_delta` (384-dim masing-masing).
2. **Clustering** — Leiden pada `topchg_delta` residual samples + Hurricane anchor.
3. **Pseudo-labeling** — mapping cluster→class berbasis v3 KMeans centroid matching (evidence CSV).
4. **Weak supervision** — ensemble 0.45·MLP + 0.35·LR + 0.20·kNN (5-fold CV, seed 42).
5. **Export** — `mlp_v3.onnx`, `lr_v3.onnx`, `knn_v3.joblib`, `scaler_v3.joblib`, `label_encoder_v3.joblib`.

---

## Catatan reproduktifitas

- Seed = 42 di semua tahap.
- Feature extraction deterministik (model pretrained frozen, tidak ada augmentasi acak saat inference).
- Ekstraksi fitur **berat** (~2 CPU core × ±5 jam untuk 3180 sampel). Sel di bawah otomatis di-skip bila cache NPZ sudah ada.
- Semua tahap juga bisa dijalankan dari CLI: `python -m src.patch_chamfer_v3`, `python -m src.cluster_v3`, `python -m src.train_v3`.

In [ ]:
import json
import os
import random
import subprocess
import sys
from pathlib import Path

import numpy as np

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
SRC_DIR = ROOT / "src"
CACHE_DIR = ROOT / "cache" / "features"
MODELS_DIR = ROOT / "models"
SUB_DIR = ROOT / "submissions"
for d in (CACHE_DIR, MODELS_DIR, SUB_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"ROOT  : {ROOT}")
print(f"Python: {sys.version.split()[0]}")

try:
    import torch
    print(f"torch : {torch.__version__} | cuda={torch.cuda.is_available()} | threads={torch.get_num_threads()}")
except Exception as e:
    print("torch not available:", e)

## 1) Patch-Chamfer feature extraction

Dijalankan hanya jika `cache/features/patch_chamfer_v3_dinov2_vits14_336.npz` belum ada.

Output NPZ berisi 6 array: `chamfer_stats`, `topchg_pre`, `topchg_post`, `topchg_delta`, `pmean_pre`, `pmean_post`.

In [ ]:
PC_PATH = CACHE_DIR / "patch_chamfer_v3_dinov2_vits14_336.npz"
RUN_EXTRACTION = not PC_PATH.exists()

if RUN_EXTRACTION:
    print("NPZ belum ada -> menjalankan ekstraksi fitur (berat, bisa berjam-jam)...")
    subprocess.run([sys.executable, "-u", "-m", "src.patch_chamfer_v3"], cwd=ROOT, check=True)
else:
    print(f"Skip ekstraksi, cache ditemukan: {PC_PATH}")

pc = np.load(PC_PATH)
print("NPZ arrays:", {k: pc[k].shape for k in pc.files})

## 2) Clustering + pseudo-label mapping

Menghasilkan `cache/features/clusters_v3.npz` (labels, hurricane_mask, residual_idx) dan `models/cluster_mapping_v3.json`.

In [ ]:
CLUSTERS_PATH = CACHE_DIR / "clusters_v3.npz"
MAPPING_PATH = MODELS_DIR / "cluster_mapping_v3.json"

if not CLUSTERS_PATH.exists() or not MAPPING_PATH.exists():
    print("Menjalankan clustering v3...")
    subprocess.run([sys.executable, "-u", "-m", "src.cluster_v3"], cwd=ROOT, check=True)
else:
    print("Skip clustering, artefak ditemukan")

clusters = np.load(CLUSTERS_PATH)
print("clusters_v3.npz:", {k: clusters[k].shape for k in clusters.files})
print(f"Hurricane anchor: {int(clusters['hurricane_mask'].sum())} sampel")

cluster_mapping = json.loads(MAPPING_PATH.read_text())
print("Cluster -> class mapping:", cluster_mapping)

## 3) Training ensemble + export artefak

Menjalankan `src.train_v3` (5-fold CV, seed 42) yang menghasilkan:

- `submissions/submission_v3.csv`
- `models/mlp_v3.onnx`, `models/lr_v3.onnx` (format model kompetisi)
- `models/knn_v3.joblib`, `models/scaler_v3.joblib`, `models/label_encoder_v3.joblib`
- `models/model_config_v3.json` (model card)

In [ ]:
subprocess.run([sys.executable, "-u", "-m", "src.train_v3"], cwd=ROOT, check=True)

for name in [
    "mlp_v3.onnx",
    "lr_v3.onnx",
    "knn_v3.joblib",
    "scaler_v3.joblib",
    "label_encoder_v3.joblib",
    "model_config_v3.json",
]:
    p = MODELS_DIR / name
    print(f"{'OK ' if p.exists() else 'MISSING '} {name:28s} {p.stat().st_size if p.exists() else 0:>10,} bytes")

In [ ]:
# Model card
card = json.loads((MODELS_DIR / "model_config_v3.json").read_text())
print(json.dumps(card, indent=2))

In [ ]:
# Sanity check: artefak ONNX bisa dimuat dan memberi prediksi
import onnxruntime as ort

pc = np.load(PC_PATH)
X = np.concatenate([pc["pmean_pre"], pc["pmean_post"], pc["topchg_delta"]], axis=1).astype(np.float32)

import joblib

scaler = joblib.load(MODELS_DIR / "scaler_v3.joblib")
le = joblib.load(MODELS_DIR / "label_encoder_v3.joblib")
Xs = scaler.transform(X).astype(np.float32)

for name in ("mlp_v3.onnx", "lr_v3.onnx"):
    sess = ort.InferenceSession(str(MODELS_DIR / name), providers=["CPUExecutionProvider"])
    proba = sess.run(None, {"input": Xs})[1]
    preds = le.classes_[np.argmax(proba, axis=1)]
    u, c = np.unique(preds, return_counts=True)
    print(name, "->", dict(zip(u.tolist(), c.tolist())))

## 4) Selesai

Model tersimpan. Untuk menghasilkan submission **tanpa retrain**, jalankan `submission_v3.ipynb`.